# 👥 View — Clientes por Região

Validação da view `vw_clientes_regiao` antes de mover para o Streamlit.

In [1]:
import pandas as pd
import sys
sys.path.append('..')

pd.set_option('display.float_format', '{:.2f}'.format)

pedidos  = pd.read_csv("../dados/pedidos_limpo.csv", parse_dates=[
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
clientes = pd.read_csv("../dados/clientes_limpo.csv")

print("Dados carregados!")

Dados carregados!


## 🧪 Testando o código antes de criar a view

In [2]:
pedidos.head(2)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13


In [3]:
clientes.head(2)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP


In [4]:
# Join entre pedidos e clientes
df = pedidos.merge(clientes, on='customer_id', how='left')

# Extraindo ano e mês
df['ano']      = df['order_purchase_timestamp'].dt.year
df['data_mes'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

# Agrupando por estado, cidade e status
clientes_regiao = (df.groupby(['customer_state', 'customer_city', 'ano', 'data_mes', 'order_status'])
                     .agg(
                         total_clientes = ('customer_unique_id', 'nunique'),
                         total_pedidos  = ('order_id',           'nunique')
                     )
                     .reset_index()
                     .sort_values('total_clientes', ascending=False))

clientes_regiao.head()

,customer_state,customer_city,ano,data_mes,order_status,total_clientes,total_pedidos
22553,SP,sao paulo,2018,2018-08,delivered,1261,1269
22537,SP,sao paulo,2018,2018-05,delivered,1176,1191
22531,SP,sao paulo,2018,2018-04,delivered,1129,1135
22525,SP,sao paulo,2018,2018-03,delivered,1111,1127
22503,SP,sao paulo,2017,2017-11,delivered,1060,1081


In [5]:
from views.vw_clientes_regiao import get_clientes_regiao

df_clientes = get_clientes_regiao(pedidos, clientes)
df_clientes.head()

,customer_state,customer_city,ano,data_mes,order_status,total_clientes,total_pedidos
22553,SP,sao paulo,2018,2018-08,delivered,1261,1269
22537,SP,sao paulo,2018,2018-05,delivered,1176,1191
22531,SP,sao paulo,2018,2018-04,delivered,1129,1135
22525,SP,sao paulo,2018,2018-03,delivered,1111,1127
22503,SP,sao paulo,2017,2017-11,delivered,1060,1081


In [6]:
# Com sum
df_clientes.groupby(['ano', 'data_mes'])['total_clientes'].sum().reset_index().head(10)

,ano,data_mes,total_clientes
0,2016,2016-09,4
1,2016,2016-10,322
2,2016,2016-12,1
3,2017,2017-01,767
4,2017,2017-02,1757
5,2017,2017-03,2643
6,2017,2017-04,2374
7,2017,2017-05,3636
8,2017,2017-06,3187
9,2017,2017-07,3956


In [ ]:
# Com max
df_clientes.groupby(['ano', 'data_mes'])['total_clientes'].max().reset_index().head(10)

,index,total_clientes
0,0,1
1,1,2
2,2,1
3,3,1
4,4,1
5,5,1
6,6,1
7,7,2
8,8,2
9,9,4
